# Axis 2: Unified Three-System Probing — Resilient Edition

Probes verb-spatial binding in **three** generative systems on AGD20K (36 affordance categories):

1. **Flux** (text-to-image) — known-good baseline replicating Zhang et al. (H2a)
2. **Cosmos-Predict2-2B-Video2World** — base video diffusion (H2b)
3. **Cosmos-Policy-ALOHA-Predict2-2B** — manipulation-fine-tuned variant (H2c)

## Required
- **Colab Pro+ recommended** (24h sessions). Plain Pro works but you'll need 2-3 sessions.
- **A100 40GB GPU** — Cosmos needs 32GB.
- **HuggingFace account** with access to FLUX.1-dev (gated) and Cosmos models.
- **AGD20K dataset** at `/content/drive/MyDrive/datasets/AGD20K` (or let cell 2 try direct download).

## Time budget (pilot, 30 samples × 36 categories per system)
| Phase | Time |
|---|---|
| Cell 0–2: install + auth + dataset | ~10 min one-time |
| Cell 3 Flux pilot (schnell, 4 steps) | ~45 min |
| Cell 4 Cosmos V2W pilot (12 steps, 17 frames) | ~3 hrs |
| Cell 5 Cosmos Policy pilot | ~3 hrs |
| Cell 6 three-way comparison | ~5 min |
| **TOTAL pilot** | **~7 hrs** |

## Resilience features
- **Resume:** every script writes per-sample CSVs incrementally. Re-running picks up where it left off.
- **Persistent HF cache** on Drive (cell 1) — models survive disconnects.
- **Periodic git push** of result CSVs (cell 0) — your progress lives on GitHub even if the runtime dies.
- **Explicit VRAM cleanup** between systems (cells 3.5, 4.5).
- **Idle-keepalive ping** (cell 7) — fires every 30s so Colab doesn't disconnect during long cells.

## If your session dies mid-run
Just open this notebook fresh, run cells 0–2 (fast, idempotent), then re-run whichever cell was running. The script will skip already-completed samples and continue.

## Cell 0 — Repo + dependencies + git auth

Idempotent: safe to re-run on resume. Set `GH_PAT` in Colab Secrets (`🔑` icon, left sidebar) for git pushes.

In [ ]:
import os, subprocess
from pathlib import Path

REPO = 'aleksantari/VLA-affordance'
BRANCH = 'nj-features'
REPO_DIR = '/content/VLA-affordance'

# Try to pull GH_PAT from Colab secrets for authenticated push
GH_PAT = ''
try:
    from google.colab import userdata
    GH_PAT = userdata.get('GH_PAT') or ''
    if GH_PAT:
        print('✓ GH_PAT found in Colab secrets — pushes enabled.')
    else:
        print('⚠ No GH_PAT in Colab secrets. Pushes will fail; results will only persist via Drive backup (cell 6 backup).')
except Exception:
    print('⚠ Colab userdata unavailable. Pushes will fail.')

remote = f'https://{GH_PAT}@github.com/{REPO}.git' if GH_PAT else f'https://github.com/{REPO}.git'

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', remote, REPO_DIR], check=True)
%cd /content/VLA-affordance
subprocess.run(['git', 'remote', 'set-url', 'origin', remote], check=True)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
subprocess.run(['git', 'config', 'user.email', 'jajda.n@northeastern.edu'], check=True)
subprocess.run(['git', 'config', 'user.name', 'nitik1998 (Colab)'], check=True)

!pip install -e . -q
!pip install -r requirements_axis2.txt -q
print('\n✓ Setup complete')

## Cell 1 — GPU check + persistent HF cache on Drive

Mounts Drive, redirects `~/.cache/huggingface` to a Drive path so the 24+ GB of model weights survive runtime disconnects.

In [ ]:
import torch, os
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime > Change runtime type > A100')
name = torch.cuda.get_device_name()
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {name}  VRAM: {vram:.1f} GB')
if vram < 35:
    print('⚠ Cosmos V2W needs ~32 GB; this GPU may OOM. Consider stronger A100/H100.')

from google.colab import drive
drive.mount('/content/drive')

# Persistent HF cache so Flux+Cosmos download once, then reuse across sessions
DRIVE_HF_CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = DRIVE_HF_CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_HF_CACHE + '/hub'
os.environ['TRANSFORMERS_CACHE'] = DRIVE_HF_CACHE + '/transformers'
print(f'✓ HF_HOME = {DRIVE_HF_CACHE}')

# HuggingFace login (gated models like FLUX.1-dev need this)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('✓ HF login OK')
    else:
        print('⚠ No HF_TOKEN in Colab secrets. Set one at hf.co/settings/tokens, add as Colab secret HF_TOKEN.')
except Exception as e:
    print(f'⚠ HF login skipped: {e}')

## Cell 2 — Persistent results dir + AGD20K

Results are written to `./results/` AND symlinked to a Drive path so they survive disconnects. AGD20K is symlinked from Drive if pre-staged there; falls back to direct download.

In [ ]:
import os, shutil
from pathlib import Path

# Mirror results to Drive: ./results -> /content/drive/MyDrive/VLA-affordance-results
DRIVE_RESULTS = Path('/content/drive/MyDrive/VLA-affordance-results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
if not Path('./results').exists() or not Path('./results').is_symlink():
    if Path('./results').exists():
        shutil.rmtree('./results')
    os.symlink(str(DRIVE_RESULTS), './results')
print(f'✓ ./results -> {DRIVE_RESULTS}')

# AGD20K — try Drive first, then direct download
DRIVE_AGD20K_PATH = '/content/drive/MyDrive/datasets/AGD20K'
if os.path.exists(DRIVE_AGD20K_PATH):
    !python data/download_agd20k.py --from_drive {DRIVE_AGD20K_PATH} --output_dir ./data/agd20k
else:
    print(f'⚠ {DRIVE_AGD20K_PATH} not found — trying direct download. This may fail if the embedded Drive ID is stale; in that case, manually download AGD20K from https://github.com/lhc1224/Cross-View-AG and put it at the path above.')
    !python data/download_agd20k.py --output_dir ./data/agd20k
!python data/download_agd20k.py --validate_only --output_dir ./data/agd20k

## Cell 3 — System A: Flux (H2a)

`--commit_every 100` pushes the per-sample CSV every 100 samples. `--resume` (default on) skips samples already in the CSV.

In [ ]:
!python scripts/09_setup_flux.py --model schnell

In [ ]:
# Pilot — 30 samples per category, schnell. Resumes from CSV if interrupted.
!python scripts/10_run_interaction_probing.py \
    --model schnell \
    --max_per_category 30 \
    --commit_every 100 \
    --save_attention_maps

In [ ]:
# Final-quality (uncomment when pilot looks good).
# Will also resume incrementally if interrupted.
# !python scripts/10_run_interaction_probing.py --model dev --commit_every 100 --save_attention_maps

## Cell 3.5 — Free Flux from VRAM before loading Cosmos

Critical: without this you'll OOM when loading Cosmos.

In [ ]:
import gc, torch
for v in list(globals()):
    if v.startswith('_') or v in ('gc', 'torch'):
        continue
    try:
        obj = globals()[v]
        if 'Pipeline' in type(obj).__name__ or 'Flux' in type(obj).__name__:
            del globals()[v]
    except Exception:
        pass
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')

## Cell 4 — System B: Cosmos-Predict2 V2W (H2b)

First-frame image conditioning with the AGD20K image itself. ~3 hours pilot. Resumes incrementally.

In [ ]:
# Quick smoke test on 3 categories × 3 samples
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_predict2_v2w \
    --categories cut hold pour \
    --max_per_category 3 \
    --num_inference_steps 8

In [ ]:
# Full pilot — 30/category. Will pick up incrementally if interrupted.
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_predict2_v2w \
    --max_per_category 30 \
    --num_inference_steps 12 \
    --commit_every 50

## Cell 4.5 — Free Cosmos V2W before loading Cosmos Policy

In [ ]:
import gc, torch
for v in list(globals()):
    if v.startswith('_') or v in ('gc', 'torch'):
        continue
    try:
        obj = globals()[v]
        if 'Pipeline' in type(obj).__name__ or 'Cosmos' in type(obj).__name__:
            del globals()[v]
    except Exception:
        pass
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')

## Cell 5 — System C: Cosmos Policy (H2c)

Manipulation-FT variant. **Same architecture as Cosmos V2W**, so VRAM is similar. Resumes incrementally.

In [ ]:
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_policy \
    --categories cut hold pour \
    --max_per_category 3 \
    --num_inference_steps 8

In [ ]:
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_policy \
    --max_per_category 30 \
    --num_inference_steps 12 \
    --commit_every 50

## Cell 6 — Three-way comparison + final push

In [ ]:
!python scripts/12_three_way_comparison.py

In [ ]:
from IPython.display import Image, display
from pathlib import Path
for f in sorted(Path('./results/figures/axis2').glob('three_way_*.png')):
    print(f.name); display(Image(str(f), width=900))
import json, pprint
with open('./results/tables/axis2_hypothesis_tests.json') as fp:
    pprint.pp(json.load(fp))

In [ ]:
# Final commit + push of comparison artifacts
!git add results/tables results/figures/axis2
!git status
!git commit -m 'research(results): Axis 2 pilot — three-way comparison artifacts' || echo 'nothing new to commit'
!git push origin nj-features || echo '⚠ push failed; results are still on Drive at /content/drive/MyDrive/VLA-affordance-results'

## Cell 7 — (Optional) Idle keepalive

Run this in parallel with the long cells above. Fires a JS heartbeat every 30s to prevent Colab's idle disconnect (~90 min). NOT a magic bullet — Colab still enforces session length caps.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function ColabKeepalive() {
  console.log('keepalive ping');
  document.querySelector('colab-toolbar-button#connect')?.click();
}
setInterval(ColabKeepalive, 30000);
'''))
print('✓ Keepalive armed (every 30s)')